# Exploring the Dataset: Building the VPN Logs

**Goal:** Understand how we go from a raw LOG configuration file to a relational database table.

This notebook walks through:
1. Loading `inet-firewall/logs/dnsmasq.log` (for loading into dns_logs table) 
3. Examining what fields each access log
4. Transforming the data into a Pandas DataFrame

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [3]:
import re
from pathlib import Path

import pandas as pd

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: /Users/rrosevearehunt/Education/DATA201/GroupProject/russellmitchell


## 1. Load the Raw log File and create Data Frame

The file `inet-firewall/logs/dnsmasq.log` 

In [5]:
opendns_log = DATASET_ROOT / "gather" / "inet-firewall" / "logs" / "dnsmasq.log"


def parse_dnsmasq_logs(file_path):
    # Regex Pattern Breakdown:
    # Group 1-3: Month, Day, Time (e.g., 'Jan', '21', '00:00:09')
    # Group 4-5: Process Name and PID (e.g., 'dnsmasq', '3468')
    # Group 6: Action (e.g., 'query[A]', 'forwarded', 'reply')
    # Group 7: Query Domain (e.g., the massive encoded subdomain string)
    # Group 8: Associated Data (Everything following the 'from', 'to', or 'is' keywords)
    log_pattern = re.compile(
        r"^([A-Z][a-z]{2})\s+(\d+)\s+(\d{2}:\d{2}:\d{2})\s+([^\[]+)\[(\d+)\]:\s+(.*?)\s+(.*?)\s+(?:from|to|is)\s+(.*)$"
    )

    parsed_data = []
    target_year = 2024  # Adjust this to the actual year the incident took place

    try:
        with open(file_path) as file:
            for line in file:
                line = line.strip()

                # Skip empty lines
                if not line:
                    continue

                match = log_pattern.search(line)
                if match:
                    month, day, time, process_name, pid, action, query_domain, associated_data = (
                        match.groups()
                    )

                    # Construct a standard timestamp string
                    timestamp_str = f"{target_year}-{month}-{int(day):02d} {time}"
                    event_timestamp = pd.to_datetime(timestamp_str, format="%Y-%b-%d %H:%M:%S")

                    parsed_data.append(
                        {
                            "event_timestamp": event_timestamp,
                            "process_name": process_name.strip(),
                            "pid": int(pid),
                            "action": action.strip(),
                            "query_domain": query_domain.strip(),
                            "associated_data": associated_data.strip(),
                        }
                    )

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None

    # Convert to DataFrame
    df = pd.DataFrame(parsed_data)
    return df


# Execute the parser
dns_df = parse_dnsmasq_logs(opendns_log)

if dns_df is not None:
    # Display the first few rows to verify the schema
    print(dns_df.head())

    # Save the structured table to a CSV for database import
    dns_df.to_csv("dnsmasq_structured.csv", index=False)
    print("\nSuccessfully parsed and saved to dnsmasq_structured.csv")

      event_timestamp process_name   pid     action  \
0 2024-01-21 00:00:09      dnsmasq  3468   query[A]   
1 2024-01-21 00:00:09      dnsmasq  3468  forwarded   
2 2024-01-21 00:00:09      dnsmasq  3468      reply   
3 2024-01-21 00:00:31      dnsmasq  3468   query[A]   
4 2024-01-21 00:00:31      dnsmasq  3468  forwarded   

                                        query_domain  associated_data  
0  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-....     10.143.0.103  
1  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-....  192.168.231.254  
2  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-....  195.128.194.168  
3  3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-....     10.143.0.103  
4  3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-....  192.168.231.254  

Successfully parsed and saved to dnsmasq_structured.csv


## 2. Identify Compromised Internal Host

In a DNS exfiltration attack, a specific internal machine will generate an abnormally massive amount of DNS queries in a very short time as it attempts to chunk and send the stolen file.

In [6]:
# Filter for only DNS queries (ignoring forwards and replies for now)
queries_df = dns_df[dns_df["action"].str.startswith("query")]

# Count the number of requests made by each internal IP
top_clients = queries_df["associated_data"].value_counts()

print("Top Requesting Internal IPs:")
print(top_clients.head(10))

Top Requesting Internal IPs:
associated_data
10.143.1.78       29956
10.143.0.103      17765
172.19.131.174    11036
172.19.130.4       7636
10.143.2.91        7532
10.143.3.65        4331
10.143.2.25        1543
172.19.130.68       588
172.19.130.106      349
10.143.2.4          232
Name: count, dtype: int64


## 3. Detect DNS Tunneling via Query Length

Standard DNS queries (like www.google.com) are short. DNS tunneling relies on packing data into the subdomain string, resulting in massive lengths (often pushing the 253-character protocol limit).


In [7]:
# Create a new column calculating the length of the queried domain
queries_df = queries_df.copy()
queries_df["domain_length"] = queries_df["query_domain"].apply(len)

# Filter for abnormally long queries (e.g., greater than 100 characters)
suspicious_queries = queries_df[queries_df["domain_length"] > 100]

print(f"Found {len(suspicious_queries)} suspiciously long DNS queries.")
print("\nSample of Exfiltration Payloads:")
print(suspicious_queries[["event_timestamp", "associated_data", "query_domain"]].head())

Found 17670 suspiciously long DNS queries.

Sample of Exfiltration Payloads:
       event_timestamp associated_data  \
0  2024-01-21 00:00:09    10.143.0.103   
3  2024-01-21 00:00:31    10.143.0.103   
6  2024-01-21 00:00:49    10.143.0.103   
9  2024-01-21 00:01:01    10.143.0.103   
12 2024-01-21 00:01:16    10.143.0.103   

                                         query_domain  
0   3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-....  
3   3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-....  
6   3x6-.598-.IzQN/wr2yUx2rxlAUPMOsynA0ZKIvumnoL-....  
9   3x6-.599-.Z*whNooy*Qten1O1tKXBeVVSzQwVo69d7I-....  
12  3x6-.600-.SLF3QepJq/D/pMU6Vg9MKwD9r0zC/43B1l-....  


## 4. Root Domain Subdomain Entropy (Finding the C2 Server)

Normal domains (like microsoft.com) might have a few dozen subdomains queried by your network (www, mail, api). Malicious exfiltration domains will have thousands of highly unique, random subdomains (because every subdomain is a chunk of the stolen file).

In [9]:
# Helper function to extract the root domain (e.g., keeping just 'kennedy-mendoza.info')
def get_root_domain(fqdn):
    parts = fqdn.split(".")
    # Simplistic extraction: grab the last two parts of the domain
    if len(parts) >= 2:
        return f"{parts[-2]}.{parts[-1]}"
    return fqdn


queries_df["root_domain"] = queries_df["query_domain"].apply(get_root_domain)

# Count how many UNIQUE full queries exist for each root domain
subdomain_entropy = (
    queries_df.groupby("root_domain")["query_domain"].nunique().sort_values(ascending=False)
)

print("Domains with the highest number of unique subdomains:")
print(subdomain_entropy.head(5))

Domains with the highest number of unique subdomains:
root_domain
kennedy-mendoza.info    17669
russellmitchell.com       138
akamaiedge.net             92
amazonaws.com              71
cloudfront.net             55
Name: query_domain, dtype: int64


## 5. Traffic Burst Analysis

Data exfiltration is rarely a slow drip; it usually happens in an automated burst.

In [11]:
# Set the timestamp as the index
time_df = queries_df.set_index("event_timestamp")

# Resample the data to count the number of queries per second ('S')
queries_per_second = time_df.resample("s").size()

# Find the seconds with the highest volume of traffic
spikes = queries_per_second.sort_values(ascending=False)

print("Peak DNS Traffic Bursts (Queries per Second):")
print(spikes.head(10))

Peak DNS Traffic Bursts (Queries per Second):
event_timestamp
2024-01-24 03:01:21    135
2024-01-21 10:45:45     96
2024-01-21 17:26:36     92
2024-01-21 05:45:20     91
2024-01-24 09:53:41     78
2024-01-23 19:32:07     75
2024-01-24 02:22:51     71
2024-01-21 06:05:22     70
2024-01-22 13:43:47     69
2024-01-24 19:34:44     67
dtype: int64


## 5. Suspicious Resolution Mapping

When an attacker controls the authoritative DNS server for the exfiltration domain, they will often configure it to reply with a single static IP, or a dummy loopback IP, because the "reply" doesn't matter to them—only receiving the query does.

In [12]:
# Filter for reply logs
replies_df = dns_df[dns_df["action"] == "reply"]

# Filter for replies resolving our suspected exfiltration domain
suspicious_replies = replies_df[
    replies_df["query_domain"].str.contains("kennedy-mendoza.info", na=False)
]

# See what IP addresses the malicious DNS server is handing back
print("IPs returned by the malicious DNS Server:")
print(suspicious_replies["associated_data"].value_counts())

IPs returned by the malicious DNS Server:
associated_data
195.128.194.168    17668
Name: count, dtype: int64


## 6. Display full table


In [14]:
# Display the full table
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", None)
dns_df

,event_timestamp,process_name,pid,action,query_domain,associated_data
0,2024-01-21 00:00:09,dnsmasq,3468,query[A],3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...,10.143.0.103
1,2024-01-21 00:00:09,dnsmasq,3468,forwarded,3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...,192.168.231.254
2,2024-01-21 00:00:09,dnsmasq,3468,reply,3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...,195.128.194.168
3,2024-01-21 00:00:31,dnsmasq,3468,query[A],3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-.u9lQ3wFEj1...,10.143.0.103
4,2024-01-21 00:00:31,dnsmasq,3468,forwarded,3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-.u9lQ3wFEj1...,192.168.231.254
...,...,...,...,...,...,...
275894,2024-01-24 23:55:43,dnsmasq,3468,query[A],e6410.d.akamaiedge.net,10.143.1.78
275895,2024-01-24 23:55:43,dnsmasq,3468,forwarded,e6410.d.akamaiedge.net,192.168.231.254
275896,2024-01-24 23:55:43,dnsmasq,3468,reply,e6410.d.akamaiedge.net,2.18.168.196
275897,2024-01-24 23:58:27,dnsmasq,3468,query[AAAA],mail,172.19.130.4


In [16]:
# Query the DataFrame with SQL (pandasql runs SQL on the in-memory table)

from pandasql import sqldf

# 1. Add the "FROM openvpn_df" to the SQL string
query = """
    SELECT event_timestamp, process_name, pid, action, query_domain, associated_data
    FROM dns_df
    ORDER BY event_timestamp ASC;
"""

# 2. Pass globals() so pandasql can locate the 'openvpn_df' variable in memory
sorted_logs_df = sqldf(query, globals())

# Print the first few rows to verify
print(sorted_logs_df.head())

              event_timestamp process_name   pid     action  \
0  2024-01-21 00:00:09.000000      dnsmasq  3468   query[A]   
1  2024-01-21 00:00:09.000000      dnsmasq  3468  forwarded   
2  2024-01-21 00:00:09.000000      dnsmasq  3468      reply   
3  2024-01-21 00:00:31.000000      dnsmasq  3468   query[A]   
4  2024-01-21 00:00:31.000000      dnsmasq  3468  forwarded   

                                                  query_domain  \
0  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...   
1  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...   
2  3x6-.596-.IunWTzebVlyAhhHj*ZfWjOBun1zAf*Wgpq-.YarqcF7oov...   
3  3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-.u9lQ3wFEj1...   
4  3x6-.597-.L**fA/ib4pGEIb5*uJ223L5A/pWGilEyrR-.u9lQ3wFEj1...   

   associated_data  
0     10.143.0.103  
1  192.168.231.254  
2  195.128.194.168  
3     10.143.0.103  
4  192.168.231.254  


## 7. Mapping to the Database Schema

Here's how this log data maps to our planned **`vpn_log`** table in PostgreSQL:

| LOG field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `dns_log_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `event_timestamp` | `event_timestamp` | `TIMESTAMP NOT NULL` | timestamp of the event|
| `process_name` | `process_name` | `VARCHAR(100)` | Process Name |
| `pid` | `pid` | `INT` | Process ID |
| `action` | `action` | `VARCHAR(100)` | Log message |
| `query_domain` | `query_domain` | `VARCHAR(255)` | domain name (e.g. www.google.com) |
| `associated_data` | `associated_data` | `INET` | ip Address in dns query |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | When this row was inserted |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE dns_events (
    dns_log_id SERIAL PRIMARY KEY,
    event_timestamp TIMESTAMP NOT NULL,
    process_name VARCHAR(100),
    pid INT,
    action VARCHAR(100),
    query_domain VARCHAR(255),
    associated_data INET,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
);
```
